# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shoriful-mynul/flyrank-assignment1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 0. Setup and data access

This notebook uses the February 2026 feature frame created from the FlyRank warehouse data contract.

The baseline is intentionally transparent and rule-based. It uses only information available in the February feature window and does not use future outcomes, labels, product decision fields, or leakage-prone trend fields.

In [4]:
import os
import numpy as np
import pandas as pd
import duckdb

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [5]:
%pip -q install duckdb huggingface_hub

In [6]:
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
}

print("Warehouse connection configured.")

Warehouse connection configured.


## 1. My rule and its reason codes

### Baseline rule

I will prioritize existing content pages that had meaningful Google Search visibility in February but showed relatively weak click-through performance for that level of exposure.

The score will combine search visibility volume with February click-through performance, while avoiding future outcomes and leakage-prone fields.

### Reason codes

- `high_visibility_low_ctr` — the page received substantial search impressions but generated relatively few clicks for that exposure.
- `high_visibility` — the page has strong search visibility and should be reviewed because it represents a meaningful opportunity.
- `limited_visibility` — the page has relatively low search exposure, so the recommendation is lower confidence.

In [7]:
feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(CASE
            WHEN report_date >= DATE '2026-02-01'
             AND report_date < DATE '2026-03-01'
            THEN gsc_impressions ELSE 0
        END) AS feb_impressions,

        SUM(CASE
            WHEN report_date >= DATE '2026-02-01'
             AND report_date < DATE '2026-03-01'
            THEN gsc_clicks ELSE 0
        END) AS feb_clicks,

        AVG(CASE
            WHEN report_date >= DATE '2026-02-01'
             AND report_date < DATE '2026-03-01'
             AND gsc_avg_position > 0
            THEN gsc_avg_position
        END) AS feb_avg_position,

        SUM(CASE
            WHEN report_date >= DATE '2026-02-01'
             AND report_date < DATE '2026-03-01'
            THEN ga4_sessions ELSE 0
        END) AS feb_sessions,

        SUM(CASE
            WHEN report_date >= DATE '2026-02-01'
             AND report_date < DATE '2026-03-01'
            THEN scroll_events ELSE 0
        END) AS feb_scroll_events

    FROM {TABLES["fact_daily"]}

    WHERE report_date >= DATE '2026-02-01'
      AND report_date < DATE '2026-03-01'

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Feature frame shape:", feature_frame.shape)
feature_frame.head()

Feature frame shape: (0, 7)


,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_avg_position,feb_sessions,feb_scroll_events


In [8]:
print("Columns:")
print(feature_frame.columns.tolist())

print("\nMissing values:")
print(feature_frame.isna().sum())

print("\nDuplicate client-content pairs:",
      feature_frame.duplicated(
          ["client_hash_id", "content_hash_id"]
      ).sum())

Columns:
['client_hash_id', 'content_hash_id', 'feb_impressions', 'feb_clicks', 'feb_avg_position', 'feb_sessions', 'feb_scroll_events']

Missing values:
client_hash_id       0
content_hash_id      0
feb_impressions      0
feb_clicks           0
feb_avg_position     0
feb_sessions         0
feb_scroll_events    0
dtype: int64

Duplicate client-content pairs: 0


### Signal Check 1 — Search Visibility Volume

**Signal:** February Google Search impressions

**Hypothesis:** Pages with more February impressions have more observable search exposure. A minimum-volume threshold helps avoid making strong prioritization decisions from very small numbers of impressions.

I will inspect impression buckets and compare their size and average click-through behavior before deciding how visibility volume should affect the baseline score.

In [9]:
signal_df = feature_frame.copy()

signal_df["feb_ctr"] = np.where(
    signal_df["feb_impressions"] > 0,
    signal_df["feb_clicks"] / signal_df["feb_impressions"],
    np.nan
)

signal_df["feb_ctr_pct"] = signal_df["feb_ctr"] * 100

print(signal_df[
    ["feb_impressions", "feb_clicks", "feb_ctr_pct"]
].describe())

       feb_impressions  feb_clicks  feb_ctr_pct
count              0.0         0.0          0.0
mean               NaN         NaN          NaN
std                NaN         NaN          NaN
min                NaN         NaN          NaN
25%                NaN         NaN          NaN
50%                NaN         NaN          NaN
75%                NaN         NaN          NaN
max                NaN         NaN          NaN


In [10]:
signal_df["impression_bucket"] = pd.cut(
    signal_df["feb_impressions"],
    bins=[-1, 99, 499, 999, 4999, np.inf],
    labels=[
        "<100",
        "100–499",
        "500–999",
        "1,000–4,999",
        "5,000+"
    ]
)

bucket_summary = (
    signal_df
    .groupby("impression_bucket", observed=False)
    .agg(
        rows=("content_hash_id", "size"),
        avg_impressions=("feb_impressions", "mean"),
        avg_ctr_pct=("feb_ctr_pct", "mean")
    )
    .reset_index()
)

bucket_summary

,impression_bucket,rows,avg_impressions,avg_ctr_pct
0,<100,0,NaN,NaN
1,100–499,0,NaN,NaN
2,500–999,0,NaN,NaN
3,"1,000–4,999",0,NaN,NaN
4,"5,000+",0,NaN,NaN


### Signal Check 1 Verdict

**Verdict: CONFIRMED / MIXED / NOT CONFIRMED**

The February impression buckets show that [describe the actual observed pattern].

This supports / does not support using impression volume as a visibility-strength signal. I therefore use impression volume as a transparent prioritization component rather than treating it as evidence that a page will definitely perform better in the future.

## 2. Build the ranked queue (writes the CSV)

The baseline score is intentionally simple and transparent.

A page receives higher priority when it combines meaningful February search visibility with relatively weak click-through performance for that visibility.

The score is a decision-support ranking, not a prediction of future traffic.

In [11]:
baseline_df = signal_df.copy()

# Minimum visibility threshold
baseline_df["visibility_eligible"] = (
    baseline_df["feb_impressions"] >= 100
)

# CTR benchmark among visibility-eligible pages
ctr_benchmark = baseline_df.loc[
    baseline_df["visibility_eligible"],
    "feb_ctr_pct"
].median()

print("CTR benchmark:", ctr_benchmark)

CTR benchmark: nan


In [12]:
baseline_df["visibility_score"] = np.log1p(
    baseline_df["feb_impressions"]
)

baseline_df["ctr_gap"] = (
    ctr_benchmark - baseline_df["feb_ctr_pct"]
)

baseline_df["ctr_gap"] = baseline_df["ctr_gap"].clip(
    lower=0
)

baseline_df["action_score"] = (
    baseline_df["visibility_score"] *
    baseline_df["ctr_gap"]
)

baseline_df["reason_code"] = np.select(
    [
        (
            baseline_df["visibility_eligible"]
            & (baseline_df["feb_ctr_pct"] < ctr_benchmark)
        ),
        baseline_df["visibility_eligible"]
    ],
    [
        "high_visibility_low_ctr",
        "high_visibility"
    ],
    default="limited_visibility"
)

baseline_df["action"] = np.select(
    [
        baseline_df["reason_code"] == "high_visibility_low_ctr",
        baseline_df["reason_code"] == "high_visibility"
    ],
    [
        "Review first",
        "Review next"
    ],
    default="Lower-priority review"
)

ranked_queue = (
    baseline_df
    .sort_values(
        ["action_score", "feb_impressions"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

ranked_queue["rank"] = (
    ranked_queue.index + 1
)

ranked_queue.head(20)

,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_avg_position,feb_sessions,feb_scroll_events,feb_ctr,feb_ctr_pct,impression_bucket,visibility_eligible,visibility_score,ctr_gap,action_score,reason_code,action,rank


In [13]:
output_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "action_score",
    "action",
    "reason_code",
    "feb_impressions",
    "feb_clicks",
    "feb_ctr_pct",
    "feb_avg_position",
    "feb_sessions",
    "feb_scroll_events"
]

ranked_output = ranked_queue[output_columns].copy()

ranked_output.head(20)

,rank,client_hash_id,content_hash_id,action_score,action,reason_code,feb_impressions,feb_clicks,feb_ctr_pct,feb_avg_position,feb_sessions,feb_scroll_events


In [14]:
output_path = "/content/baseline_action_score.csv"

ranked_output.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows:", len(ranked_output))

Saved: /content/baseline_action_score.csv
Rows: 0


In [15]:
import os

os.makedirs("/content/work/outputs", exist_ok=True)

repo_output_path = "/content/work/outputs/baseline_action_score.csv"

ranked_output.to_csv(
    repo_output_path,
    index=False
)

print("Saved:", repo_output_path)

Saved: /content/work/outputs/baseline_action_score.csv


## 3. Top-10 review

The table below contains the 10 highest-ranked pages from the transparent baseline.

For each item, the review records:
- the recommended action,
- the reason code,
- a confidence note,
- and what evidence could make the recommendation wrong.

The ranking is intended for review prioritization, not as a guarantee that refreshing a page will improve future performance.

In [16]:
top10 = ranked_queue.head(10).copy()

top10_review = top10[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action",
        "reason_code",
        "action_score",
        "feb_impressions",
        "feb_clicks",
        "feb_ctr_pct",
        "feb_avg_position"
    ]
].copy()

top10_review

,rank,client_hash_id,content_hash_id,action,reason_code,action_score,feb_impressions,feb_clicks,feb_ctr_pct,feb_avg_position


### Top-10 review notes

**Confidence note:** Higher confidence is assigned to pages with substantial February impression volume because the CTR signal is based on more observed search exposure.

**What would make the recommendation wrong:** A high score does not prove that the page is actually a good refresh candidate. The page may have strong search intent alignment, seasonal demand, technical limitations, brand-driven search behavior, or content-quality factors that are not represented in this baseline.

In [17]:
top10[[
    "rank",
    "content_hash_id",
    "action_score",
    "feb_impressions",
    "feb_ctr_pct",
    "feb_avg_position",
    "reason_code"
]].tail(5)

,rank,content_hash_id,action_score,feb_impressions,feb_ctr_pct,feb_avg_position,reason_code


## 4. Weak picks + leakage check

### Weak picks

The weakest top-20 picks are the items where the baseline may be over-prioritizing a page because of high impression volume despite limited supporting evidence from the available February features.

These cases should be treated as review candidates rather than automatic refresh decisions.

### Leakage check

The baseline uses only February feature-window fields:
- February impressions
- February clicks
- February CTR derived from those fields
- February average position
- February sessions
- February scroll events

It does not use future-window outcomes, decline labels, `trend_direction`, `trend_pct`, or product decision fields.

In [18]:
forbidden_terms = [
    "label",
    "trend",
    "future",
    "outcome",
    "decision",
    "product"
]

used_columns = set(ranked_queue.columns)

leakage_candidates = [
    col for col in used_columns
    if any(term in col.lower() for term in forbidden_terms)
]

print("Potential leakage-related columns present:")
print(leakage_candidates)


Potential leakage-related columns present:
[]


In [19]:
required_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "action_score",
    "action",
    "reason_code"
]

print("Missing required columns:",
      [c for c in required_columns if c not in ranked_output.columns])

print("Total ranked rows:", len(ranked_output))

print("Top-20 rows:", len(ranked_output.head(20)))

print("Duplicate content-client pairs:",
      ranked_output.duplicated(
          ["client_hash_id", "content_hash_id"]
      ).sum())

print("\nReason-code counts:")
print(ranked_output["reason_code"].value_counts())

print("\nAction counts:")
print(ranked_output["action"].value_counts())

Missing required columns: []
Total ranked rows: 0
Top-20 rows: 0
Duplicate content-client pairs: 0

Reason-code counts:
Series([], Name: count, dtype: int64)

Action counts:
Series([], Name: count, dtype: int64)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.